# Ejercicio 2 — Wine Quality: ACP, t-SNE y UMAP
**Lead University · Minería de Datos · Tarea 5**

Comparación de ACP, t-SNE y UMAP sobre el dataset `winequality.csv`:  
1 599 muestras de vino tinto "Vinho Verde", 11 variables fisicoquímicas + `calidad`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from scripts import DimReducer

print('Librerías cargadas.')

Librerías cargadas.


---
## a) Carga de datos

In [2]:
df = pd.read_csv('datos/winequality.csv')
print(f'Dimensiones: {df.shape}')
print(f'Variables: {df.columns.tolist()}')
print(f'\nDistribución de calidad:')
print(df['calidad'].value_counts().sort_index())

Dimensiones: (1599, 12)
Variables: ['fija.acidez', 'volatil.acidez', 'citrica.acidez', 'residual.azucar', 'cloruros', 'libre.sulfuro.dioxido', 'total.sulfuro.dioxido', 'densidad', 'pH', 'sulfitos', 'alcohol', 'calidad']

Distribución de calidad:
calidad
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


In [3]:
df.describe().round(3)

,fija.acidez,volatil.acidez,citrica.acidez,residual.azucar,cloruros,libre.sulfuro.dioxido,total.sulfuro.dioxido,densidad,pH,sulfitos,alcohol,calidad
count,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000,1599.000
mean,8.320,0.528,0.271,2.539,0.087,15.875,46.468,0.997,3.311,0.658,10.423,5.636
std,1.741,0.179,0.195,1.410,0.047,10.460,32.895,0.002,0.154,0.170,1.066,0.808
min,4.600,0.120,0.000,0.900,0.012,1.000,6.000,0.990,2.740,0.330,8.400,3.000
25%,7.100,0.390,0.090,1.900,0.070,7.000,22.000,0.996,3.210,0.550,9.500,5.000
50%,7.900,0.520,0.260,2.200,0.079,14.000,38.000,0.997,3.310,0.620,10.200,6.000
75%,9.200,0.640,0.420,2.600,0.090,21.000,62.000,0.998,3.400,0.730,11.100,6.000
max,15.900,1.580,1.000,15.500,0.611,72.000,289.000,1.004,4.010,2.000,14.900,8.000


**Contexto del dataset:**  
Muestras del vino tinto "Vinho Verde" de Portugal. Las 11 variables predictoras son fisicoquímicas
(acidez, pH, alcohol, sulfatos, etc.) y `calidad` es una puntuación sensorial entre 3 y 8.
La relación entre propiedades fisicoquímicas y calidad es no lineal, lo que hace relevante
comparar ACP con t-SNE y UMAP.

---
## b) ACP, t-SNE y UMAP con 3 componentes

### Exploración de n_neighbors para UMAP

In [4]:
# color_col='calidad' colorea los puntos pero no entra al modelo
dr = DimReducer(df, color_col='calidad', seed=42)

dr.explore_umap_neighbors(neighbors_list=[10, 20, 30, 50])

**Selección de n_neighbors:**  
`n_neighbors=20` produce la representación más clara: gradiente de calidad visible sin que
los clústeres queden demasiado fragmentados ni difusos. **Se selecciona n_neighbors = 20.**

In [5]:
dr.fit(n_components=3, tsne_perplexity=30, umap_n_neighbors=20)

Ajuste completado — n_components=3, perplexity=30, n_neighbors=20


---
## c) Gráficos proyectados sobre las dos primeras componentes

In [6]:
dr.plot_mapa_interactivo(dr.coords_pca, title='Wine Quality — ACP')
dr.plot_mapa_interactivo(dr.coords_tsne, title='Wine Quality — t-SNE')
dr.plot_mapa_interactivo(dr.coords_umap, title='Wine Quality — UMAP')

In [7]:
dr.plot_comparacion()

---
## d) Comparación e interpretación

**Nota sobre la distribución de calidad:** El dataset está fuertemente concentrado en calidades
5 y 6 (681 y 638 muestras respectivamente), mientras que las calidades extremas son muy escasas
(calidad 3: 10 muestras, calidad 8: 18 muestras). Esto limita la separación visual posible:
los extremos son tan pocos puntos que cualquier agrupación observada debe interpretarse con cautela.

**ACP — Proyección lineal:**  
Los vinos se distribuyen en una nube continua sin agrupaciones claras por calidad.
El ACP captura las direcciones de mayor varianza fisicoquímica, que no corresponden
necesariamente a diferencias en la puntuación sensorial. La dispersión es alta y
los niveles de calidad se solapan en todo el plano.

**t-SNE — Preservación de vecindades locales:**  
Genera agrupaciones más compactas. Dado el desbalance (la mayoría son calidad 5-6),
la mayor parte de los puntos forma una masa central densa. Las calidades extremas
(3-4 y 7-8), al ser pocas muestras, pueden aparecer en regiones distintas simplemente
por su escasez, no necesariamente por una separación real del método.

**UMAP — Balance local/global:**  
Preserva tanto estructura local como global. Se puede observar cierto gradiente de calidad
más ordenado que en ACP, aunque el solapamiento entre calidades adyacentes (5 y 6) es
inevitable dado que comparten perfiles fisicoquímicos muy similares y representan el 82%
de los datos.

**Diferencias en dispersión:**  
- **ACP:** alta dispersión sin estructura, nube elíptica continua.
- **t-SNE:** dispersión reducida, micro-grupos compactos pero condicionados por el desbalance.
- **UMAP:** dispersión intermedia con el gradiente de calidad más legible de los tres.

**Conclusión:** Ningún método logra separar nítidamente los niveles de calidad, lo cual es
esperable: la calidad es una variable ordinal subjetiva con alta superposición entre clases
adyacentes y muy pocos casos en los extremos.